In [1]:
# MIMIC-IV
from pathlib import Path

import json
import polars as pl
import time
import datetime

In [2]:
local_name = "mimic-demo"
ricu_name = "mimic-demo"
version = "2.2"

In [3]:
# Paths
## Directories
data_dir = Path("/workspaces/example/data")
shared_dir = Path("/workspaces/OpenICU.example/custom/shared")

## Files
ricu_cnpt_dict_path = data_dir / "projects" / "ricu" / "inst" / "extdata" / "config" /  "concept-dict.json"
ricu_data_src_path = data_dir / "projects" / "ricu" / "inst" / "extdata" / "config" / "data-sources.json"

d_items_path = data_dir / "physionet.org" / "files" / local_name / version / "icu" / "d_items.csv.gz"
d_labitems_path = data_dir / "physionet.org" / "files" / local_name / version / "hosp" / "d_labitems.csv.gz"

In [4]:
# Json, dataframes and lazyframes

ricu_cnpt_dict_json = None
ricu_data_src_json = None

d_items_lf = None
d_labitems_lf = None

In [5]:
# ricu concept dict

ricu_cnpt_dict_df = pl.read_json(ricu_cnpt_dict_path)
with open(ricu_data_src_path, "r", encoding="utf-8") as f:
    ricu_data_src_data = json.load(f)
# miiv data
miiv_d_items_lf = pl.scan_csv(f"/workspaces/example/data/physionet.org/files/{local_name}/{version}/icu/d_items.csv.gz").with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))
miiv_d_labitems_lf = pl.scan_csv(f"/workspaces/example/data/physionet.org/files/{local_name}/{version}/hosp/d_labitems.csv.gz").with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))

In [6]:
for e in ricu_data_src_data:
    if e["name"] == ricu_name:
        tables = e["tables"]



In [7]:
type_clses = {
    "lgl_cncpt": "boolean"
}

In [8]:
def ricu_cnpt_dict_to_cust_cnpt_dict(info: pl.DataFrame, src_name: str, src_items: dict[str, pl.LazyFrame], src_item_col_name: str) -> tuple[dict, dict]:
    data: dict[str, dict] = {}
    un_src_data: dict = {}
    i = 0
    for cnpt in info.columns:
        if(cnpt_srcs := info[cnpt][0].get("sources")) and cnpt_srcs.get(src_name):
            for i in range(len(cnpt_srcs[src_name])):
                src = cnpt_srcs[src_name][i]
                cnpt_ids = src.get("ids")
                cnpt_regex = src.get("regex")
                if cnpt_ids is None and cnpt_regex is None:
                    un_src_data[cnpt] = info[cnpt]
                    continue
                cnpt_entr = info[cnpt][0]
                cnpt_cat = cnpt_entr["category"].replace(" ", "_").replace("/", "_per_")
                cnpt_name = cnpt_entr["description"].replace(" ", "_").replace("/", "_per_")
                cnpt_type_cls = cnpt_entr.get("class")
                cnpt_tbl = src["table"]
                sub_var = src["sub_var"] 

                if not data.get(cnpt_cat):
                    data[cnpt_cat] = {}

                if isinstance(cnpt_ids, str) or (isinstance(cnpt_ids, list) and isinstance(cnpt_ids[0], str)):
                    un_src_data[cnpt] = info[cnpt]
                    continue
                
                if cnpt_ids is not None:
                    cnpt_ids: list[int] = cnpt_ids if isinstance(cnpt_ids, list) else [cnpt_ids]

                if data[cnpt_cat].get(cnpt_name) is None:
                    data[cnpt_cat][cnpt_name] = []

                if isinstance((time_vars := tables[cnpt_tbl]["defaults"]["time_vars"]), list):
                    time = time_vars[0]
                else:

                    time = time_vars 
                unit = cnpt_entr.get("unit")

                data[cnpt_cat][cnpt_name].append({
                    "name": cnpt_name,
                    "unit": unit if unit is not None else type_clses.get(cnpt_type_cls),
                    "table": cnpt_tbl,
                    "code": None,
                    "ids": cnpt_ids,
                    "src_regex": cnpt_regex,
                    "short": cnpt,
                    "min": cnpt_entr.get("min"),
                    "max": cnpt_entr.get("max"),
                    "sub_var": sub_var,
                    "time": time,
                })
                if cnpt_ids is not None:
                    filtered_src_items = src_items[cnpt_tbl if cnpt_tbl in src_items.keys() else "else"].filter(pl.col(src_item_col_name).is_in(cnpt_ids)).select("code").collect()
                    regex = "|".join(list(filtered_src_items["code"]))
                else:
                    regex = cnpt_regex
                regex = regex.replace("#", "\\#")

                data[cnpt_cat][cnpt_name][i]["code"] = "(" + regex + ")"

                if data[cnpt_cat][cnpt_name][i]["max"] is None:
                    data[cnpt_cat][cnpt_name][i]["max"] = "Inf"

                if data[cnpt_cat][cnpt_name][i]["min"] is None:
                    data[cnpt_cat][cnpt_name][i]["min"] = "-Inf"

    return (data, un_src_data)

In [9]:
data_by_cat = {}
un_src_data = {}

src_item_mapping = {"labevents": miiv_d_labitems_lf, "else": miiv_d_items_lf}
(data_by_cat, un_src_data) = ricu_cnpt_dict_to_cust_cnpt_dict(ricu_cnpt_dict_df, ricu_name, src_item_mapping, "itemid")

In [10]:
data_by_cnpt = {}
for cat in data_by_cat.keys():
    for cnpt in data_by_cat[cat]:
        data_by_cnpt[cnpt] = data_by_cat[cat][cnpt]

In [11]:
# Overview about usable data
print("total category count:", len(data_by_cat))
for key in data_by_cat.keys():
    print("\tcategory:", key, "concept count:", len(data_by_cat[key]))
print("total concept count:", len(data_by_cnpt))

total category count: 0
total concept count: 0


In [12]:
# Overview about unused data
print("total unused concept count:", len(un_src_data))


total unused concept count: 0


In [13]:
with open(f"//workspaces//OpenICU.example//custom//shared//{ricu_name}//data.json", "w") as file:
    json.dump(data_by_cat, file, indent=4, ensure_ascii=False)

In [14]:
def create_general_cnpt(dir_path: str | Path, name: str, unit: str, table_contents: list[dict]) -> None:
    if isinstance(unit, list):
        unit = unit[0]
    if unit == "%":
        unit = "\"%\""

    path = Path(dir_path)
    path.mkdir(parents=True, exist_ok=True)
    print(path)
    out_file = path / f"{name}.yml"

    # Existenz prüfen
    if out_file.exists():
        print(f"File already exists: {out_file}")
        return

    content = f"""name: {name}
version: 1.0.0
unit: {unit}
extension_columns:
  dataset: col("dataset")
  table: col("table")
"""

    with open(out_file, "w", encoding="utf8") as f:
        f.write(content)

    print(f"Created: {out_file}")

In [15]:
def create_src_specific_cnpt(dir_path: str | Path, name: str, unit: str, table_contents: list[dict]) -> None:
    if isinstance(unit, list):
        unit = unit[0]
    if unit == "%":
        unit = "\"%\""

    path = Path(dir_path)
    path.mkdir(parents=True, exist_ok=True)
    print(path)
    out_file = path / f"{name}.yml"

    # Existenz prüfen
    if out_file.exists():
        print(f"File already exists: {out_file}")
        return

    content = """type: simple
mappings:
"""
    for table_content in table_contents:
      content +=  f"""  - pattern:
      table: {table_content["table"]}
      event: result
      code: {table_content["code"]}
    columns:
      numeric_value: col(numeric_value)
      text_value: col(text_value)
"""

    with open(out_file, "w", encoding="utf8") as f:
        f.write(content)

    print(f"Created: {out_file}")

In [16]:
timestamp = datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d_%H-%M-%S')
base_path = Path(f"/workspaces/OpenICU.example/custom/generated/temp/{ricu_name}/config/{timestamp}/{ricu_name}/{version}/concept")

for category_name in data_by_cat.keys():
    # folder = base_path / category_name.replace(" ", "_").replace("/", "_per_")
    folder = base_path
    folder.mkdir(parents=True, exist_ok=True)
    for cnpt in (category := data_by_cat[category_name]).keys() :
        create_src_specific_cnpt(folder,
        category[cnpt][0]["name"].replace(" ", "_").replace("/", "_per_"),
        category[cnpt][0]["unit"] if isinstance(category[cnpt][0]["unit"], list) else category[cnpt][0]["unit"] ,
        [{"table": category[cnpt][i]["table"], "code": category[cnpt][i]["code"]} for i in range(len(category[cnpt]))]
        )
        create_general_cnpt(Path(f"/workspaces/OpenICU.example/custom/generated/temp/{ricu_name}/config/{timestamp}/concept") , category[cnpt][0]["name"].replace(" ", "_").replace("/", "_per_"),
        category[cnpt][0]["unit"] if isinstance(category[cnpt][0]["unit"], list) else category[cnpt][0]["unit"] ,
        [{"table": category[cnpt][i]["table"], "code": category[cnpt][i]["code"]} for i in range(len(category[cnpt]))])

        # create_general_cnpt(Path(f"/workspaces/OpenICU.example/custom/generated/temp/{ricu_name}/config/{timestamp}/concept") / category_name.replace(" ", "_").replace("/", "_per_"), category[cnpt][0]["name"].replace(" ", "_").replace("/", "_per_"),
        # category[cnpt][0]["unit"] if isinstance(category[cnpt][0]["unit"], list) else category[cnpt][0]["unit"] ,
        # [{"table": category[cnpt][i]["table"], "code": category[cnpt][i]["code"]} for i in range(len(category[cnpt]))])
    

In [17]:
# Run pipeline.ipynb

# Sanity check

In [18]:
# Prepare needed information to make check inside of ricu/ R

# Root folder were the concepts are saved
cnpt_root_folder = Path("/workspaces/OpenICU.example/output/project/workspace/concept")

found_data_info = {}

# iterate all the folders in the root folder
for cnpt_folder in cnpt_root_folder.iterdir():
    # if the cnpt_folder is a folder/directory execute logic
    if cnpt_folder.is_dir():
        # select the first file in the sub folder "/1.0.0/" and exprect it to be the cnpt_file
        cnpt_file  = next((file for file in (cnpt_folder / "1.0.0").iterdir() if file.is_file()), None)
        print(cnpt_file)
        if cnpt_file is None:
            print(f"{cnpt_folder.name}: empty")
            continue  # or size = 0, depending on your logic
        # print(f"{cnpt_folder}: {cnpt_file.name if cnpt_file else "empty"}")
        size = len(pl.read_parquet(cnpt_file))
        # print(size)
        name = cnpt_folder.name
        if data_by_cnpt.get(name):
            found_data_info[name] = data_by_cnpt[name]
        else:
            print("Not found: ", name)

FileNotFoundError: [Errno 2] No such file or directory: '/workspaces/OpenICU.example/output/project/workspace/concept'

In [ ]:
found_data_info
data_by_cnpt

In [ ]:
with open(f"/workspaces//OpenICU.example//custom//shared//{ricu_name}//check-open_icu-to-ricu-{datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d_%H-%M-%S')}.json", "w") as file:
    json.dump(found_data_info, file, indent=4, ensure_ascii=False)
# found_data_info

In [ ]:
print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d_%H-%M-%S'))

## Make Checklist for Github

In [ ]:

def make_issue_checklist(ricu: pl.DataFrame):
    cat = {}

    for col in ricu.columns:
        elem = list(ricu[col])[0]
        if not cat.get(elem["category"]):
            cat[elem["category"]] = []
        cat[elem["category"]] += [elem["description"]]
    output = ""
    for cat, cnpts in cat.items():
        output += f"- [ ] {cat}:\n"
        for cnpt in cnpts:
            output += ((f"  - [x] {cnpt}\n") if (cnpt.replace(" ", "_").replace("/", "_per_") in data_by_cnpt.keys()) else (f"  - [ ] {cnpt}\n"))
            if (cnpt.replace(" ", "_").replace("/", "_per_") not in data_by_cnpt.keys()):
                print(f"{cnpt}")
    return output

In [ ]:
output = make_issue_checklist(ricu_cnpt_dict_df)